# Stage 3: normalize + log1p + highly variable genes (HVG)

Standard scanpy normalization pipeline on the filtered count matrix:

1. **Preserve raw counts** in `adata.layers['counts']` for downstream methods that
   need them (scVI, scANVI, DESeq2, pseudobulk). This is a copy of the sparse CSR matrix.
2. **Library-size normalization** to 10,000 counts per cell.
3. **Log1p transform** for variance stabilization.
4. **HVG selection** — identify the most variable genes for dimensionality reduction.
5. **Float32 cast** — Memory Discipline #5, saves 2x memory vs float64 with no meaningful precision loss.

**What this notebook produces**:
- `adata.layers['counts']` — raw counts preserved for downstream methods
- `adata.X` — normalized, log1p-transformed, float32
- `adata.var['highly_variable']` — HVG boolean mask
- `adata.uns['normalize_v1']` — normalization parameters recorded
- Stage 3 checkpoint `.h5ad` ready for stage 4 embedding

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  — stage 2 (post-filter) output.
# OUTPUT_PATH    — where to write this stage's checkpoint.
# n_top_genes    — number of HVGs to select.
# HVG_FLAVOR     — HVG method ("seurat" / "seurat_v3" / "cell_ranger").
#                   seurat (default) works on log-normalized data;
#                   seurat_v3 expects raw counts (use layers['counts']).

UPSTREAM_PATH = "results/nancang_stage2_qcd_v1.h5ad"
OUTPUT_PATH   = "results/nancang_stage3_normalized_v1.h5ad"

N_TOP_GENES   = 2000    # number of highly variable genes
HVG_FLAVOR    = "seurat"  # "seurat" | "seurat_v3" | "cell_ranger"

In [ ]:
# Ensure the framework src/ is on sys.path and CWD is set to the project root.
# Detects: if running from notebooks/ (Jupyter) or from project root (nbconvert).
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")


In [ ]:
# Imports + load stage 2 checkpoint.
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import datetime

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

print("Loading upstream:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

In [ ]:
# Preserve raw counts in adata.layers['counts'].  These are needed by scVI, scANVI,
# DESeq2, and pseudobulk methods that operate on raw count data.
# This is a copy of the sparse CSR matrix — memory doubles at this point,
# which is intentional; the copy is explicit and traceable.
print("Copying raw counts to adata.layers['counts']...")
adata.layers['counts'] = adata.X.copy()
print(f"layers keys: {list(adata.layers.keys())}")

In [ ]:
# Library-size normalization: scale each cell to 10,000 total counts.
# This is the standard scRNA-seq normalization that accounts for
# differences in sequencing depth across cells.
print("\nNormalizing total counts per cell (target_sum=1e4)...")
sc.pp.normalize_total(adata, target_sum=1e4)
print(f"After normalize_total: X mean={adata.X.mean():.2f}, max={adata.X.max():.0f}")

In [ ]:
# Log1p transform: log(1 + x) for variance stabilization.
# Transforms the count data from a discrete, skewed distribution
# to a more continuous, approximately normal distribution suitable
# for PCA and other linear methods.
print("\nApplying log1p transform...")
sc.pp.log1p(adata)
print(f"After log1p: X mean={adata.X.mean():.4f}")

In [ ]:
# Highly variable gene (HVG) selection.
# HVGs carry the most biological signal; keeping them for dimensionality
# reduction reduces noise and memory footprint.
print(f"\nSelecting top {N_TOP_GENES} HVGs (flavor={HVG_FLAVOR})...")
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_TOP_GENES,
    flavor=HVG_FLAVOR,
)
n_hvg = adata.var['highly_variable'].sum()
print(f"HVGs selected: {n_hvg}")

In [ ]:
# Cast to float32 — Memory Discipline #5.
# scRNA-seq normalized counts and embeddings carry far less than float64's
# 15 decimal digits of precision. float64 doubles memory for no useful gain.
print("\nCasting adata.X to float32 (Memory Discipline #5)...")
adata.X = adata.X.astype(np.float32)
print(f"X dtype now: {adata.X.dtype}")

In [ ]:
# HVG diagnostic plot: mean expression vs dispersion, with HVGs highlighted.
# PI inspects this to confirm the HVG selection captured the expected
# highly-expressed variable genes and didn't over-select low-expression noise.
sc.pl.highly_variable_genes(adata, save=".png")
# sc.pl.highly_variable_genes in scanpy >=1.10 uses its own figure;
# the saved plot goes to ./figures/highly_variable_genes.png
import os, shutil
fig_src = "figures/highly_variable_genes.png"
fig_dst = "results/figures/stage3_hvg.png"
if os.path.exists(fig_src):
    shutil.move(fig_src, fig_dst)
    print(f"HVG plot saved to {fig_dst}")


In [ ]:
# Record normalization parameters — plain adata.uns write (SPEC Run Metadata).
# Versioned key (normalize_v1) allows future re-runs with different parameters
# to coexist for comparison.
adata.uns['normalize_v1'] = {
    "target_sum":  1e4,
    "log_transformed": True,
    "hvg_flavor":  HVG_FLAVOR,
    "n_top_genes": N_TOP_GENES,
    "n_hvg":       n_hvg,
    "timestamp":   datetime.datetime.now().isoformat(),
}

# Also record layer origin for traceability.
adata.uns["counts_layer"] = {
    "name": "counts",
    "description": "Raw counts preserved from stage 2 post-filter adata.X (before normalize_total)",
    "source": UPSTREAM_PATH,
}

print("normalize_v1:", adata.uns['normalize_v1'])

In [ ]:
# Memory discipline self-check (one assertion before write — SPEC Memory Discipline).
# Guards the highest-impact memory regression: adata.X becoming dense or losing float32.
# If this ever fails, investigate which upstream operation densified or cast the matrix.
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# Write the stage checkpoint to disk.
# compression="lzf" is Memory Discipline #4 — faster than gzip,
# ~30% smaller than uncompressed, and preserves sparse CSR layout.
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

# Verify the file was written and is readable.
import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# Free memory across stage boundaries (Memory Discipline #3).
# Without this, the Jupyter kernel keeps the previous stage's AnnData
# alive when the next stage is run in the same kernel session.
del adata
import gc
gc.collect()
print("Memory released.")